<a href="https://colab.research.google.com/github/imannolM/2250-IA-para-Desarrolladores-Desarrollando-Coodigos-con-ChatGPT-Grok-Claude-y-Gemini/blob/main/agentes_con_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd

df = pd.read_csv('/content/datos_entregas.csv')

df.head()

,ID_pedido,años_experiencia_colaborador,clasificacion_colaborador,latitud_tienda,longitud_tienda,latitud_entrega,longitud_entrega,fecha_pedido,hora_pedido,hora_retirada,clima,trafico,vehiculo,area,categoria_producto,tiempo_entrega
0,ialx566343618,8,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Soleado,Alto,Motocicleta,Urbano,Ropa,120
1,akqg208421122,10,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Lluvioso,Congestion,Scooter,Metropolitano,Electrónicos,165
2,njpu434582536,16,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Tormenta,Bajo,Motocicleta,Urbano,Deportes,130
3,rjto796129700,8,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Soleado,Medio,Motocicleta,Metropolitano,Cosméticos,105
4,zguw716275638,11,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Nublado,Alto,Scooter,Metropolitano,Juguetes,150


In [12]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [13]:
!pip install langchain-groq -q

In [29]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key=GROQ_API_KEY, model_name = 'llama-3.3-70b-versatile', temperature = 0)

In [30]:
ai_msg = llm.invoke("""
                    Tengo un dataframe llamado 'df' con las columnas 'años_experiencia_colaborador' y 'tiempo_entrega'.
                    Escribe el código Python con la biblioteca pandas para calcular la correlacion entre las dos columnas.
                    Devuelve la respuesta en Markdown para el trecho de código Python únicamente.
                    """)

print(ai_msg)

content="```python\nimport pandas as pd\n\n# Calcula la correlación entre las columnas 'años_experiencia_colaborador' y 'tiempo_entrega'\ncorrelacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])\n\nprint(correlacion)\n```" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 107, 'total_tokens': 172, 'completion_time': 0.152185101, 'completion_tokens_details': None, 'prompt_time': 0.015517087, 'prompt_tokens_details': None, 'queue_time': 0.134766275, 'total_time': 0.167702188}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f54e8-e3da-7fa3-b14f-dfab20a08303-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 107, 'output_tokens': 65, 'total_tokens': 172}


In [31]:
print(ai_msg.content)

```python
import pandas as pd

# Calcula la correlación entre las columnas 'años_experiencia_colaborador' y 'tiempo_entrega'
correlacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])

print(correlacion)
```


In [32]:
correlacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])
print(f"La correlación entre años de experiencia y tiempo de entrega es: {correlacion}")


La correlación entre años de experiencia y tiempo de entrega es: -0.2516410755060904


PythonAstREPLTool Lee-Evalua-Imprime-Itera

In [33]:
!pip install langchain-experimental -q

In [63]:
from langchain_experimental.tools import PythonAstREPLTool

herramienta_py = PythonAstREPLTool(locals={"df":df})
herramienta_py.invoke("df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])")


np.float64(-0.2516410755060904)

In [39]:
llm_con_herramienta = llm.bind_tools([herramienta_py],tool_choice=herramienta_py.name)
respuesta = llm_con_herramienta.invoke( """
                                       Tengo un dataframe llamado 'df' con las columnas 'años_experiencia_colaborador' y 'tiempo_entrega'.
                                       Escribe el código Python con la biblioteca pandas para calcular la correlacion entre las dos columnas.
                                       Devuelve la respuesta en Markdown para el trecho de código Python únicamente, separando siempre cada
                                       comando utilizando ';'.
                                       """)
respuesta.tool_calls

[{'name': 'python_repl_ast',
  'args': {'query': "import pandas as pd; df = pd.DataFrame({'años_experiencia_colaborador': [1, 2, 3, 4, 5], 'tiempo_entrega': [10, 15, 7, 12, 8]}); correlacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega']); print(correlacion)"},
  'id': 'tjrn694mr',
  'type': 'tool_call'}]

In [41]:
from langchain_core.output_parsers.openai_tools import JsonOutputKeyToolsParser

parser = JsonOutputKeyToolsParser(key_name=herramienta_py.name, first_tool_only=True)

cadena = llm_con_herramienta | parser

respuesta = cadena.invoke( """
              Tengo un dataframe llamado 'df' con las columnas 'años_experiencia_colaborador' y 'tiempo_entrega'.
              Escribe el código Python con la biblioteca pandas para calcular la correlacion entre las dos columnas.
              Devuelve la respuesta en Markdown para el trecho de código Python únicamente, separando siempre cada
              comando utilizando ';'.
              """)
respuesta["query"]

"import pandas as pd; df = pd.DataFrame({'años_experiencia_colaborador': [1, 2, 3, 4, 5], 'tiempo_entrega': [10, 8, 12, 9, 11]}); correlacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega']); print(correlacion)"

In [43]:
import pandas as pd; df = pd.DataFrame({'años_experiencia_colaborador': [1, 2, 3, 4, 5], 'tiempo_entrega': [10, 8, 12, 9, 11]}); correlacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega']); print(correlacion)

0.3


In [69]:
df = pd.read_csv('/content/datos_entregas.csv')
df.columns.to_list()
herramienta_py = PythonAstREPLTool(locals={"df": df})

In [70]:
from langchain_core.prompts import ChatPromptTemplate

system = f"""
          Tienes acceso a un dataframe pandas `df`.
          Aquí esta la salida del comando 'df.head().to_markdown()':
          ```md
          {df.columns.to_list()}
          ```
          Dada una pregunta de usuario, escribe el codigo Python para responderla,
          cada comando Python generado SIEMPRE debe estar separado por ';'.
          Los únicos recursos que debes utilizar son la bibliotecas incorporadas Python y pandas.
          Retorna ÚNICAMENTE el código Python.
          """

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}")
    ]
)

In [71]:
cadena = prompt | llm_con_herramienta | parser | herramienta_py

#respuesta = cadena.invoke({"question":"¿Cuál es la correlación entre los años de experiencia del colaborador y el tiempo de entrega?"})

#print(respuesta)

In [72]:
respuesta = cadena.invoke({"question":"¿Cuál es el promedio de tiempo de entrega para cada clima?"})

print(respuesta)

clima
Lluvioso    123.211825
Nublado     138.286773
Soleado     103.664453
Tormenta    123.238509
Name: tiempo_entrega, dtype: float64


In [90]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import ToolMessage

system = f"""
          Tienes acceso a un dataframe pandas `df`.
          Aquí esta la salida del comando 'df.head().to_markdown()':
          ```md
          {df.columns.to_list()}
          ```
          Dada una pregunta de usuario, escribe el codigo Python para responderla,
          cada comando Python generado SIEMPRE debe estar separado por ';'.
          Los únicos recursos que debes utilizar son la bibliotecas incorporadas Python y pandas.
          Finalmente, responde en español la pregunta cuando tengas la información suficiente para hacerlo.
          """

def _get_chat_history(x: dict) -> list:
  """
  Analiza la salida de la cadena hasta este punto en una lista de mensajes
  del historial del chat para insertar en el prompt.
  """
  ai_msg = x['ai_msg']
  tool_call_id = x['ai_msg'].additional_kwargs['tool_calls'][0]['id']
  tool_msg = ToolMessage(tool_call_id=tool_call_id, content=str(x['tool_output']))
  return [ai_msg, tool_msg]

In [91]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

cadena = (
    RunnablePassthrough.assign(ai_msg = prompt | llm_con_herramienta)\
                       .assign(tool_output = itemgetter("ai_msg") | parser | herramienta_py)\
                       .assign(chat_history = _get_chat_history)\
                       .assign(response = prompt | llm | StrOutputParser())\
                       .pick(["tool_output","response"])
)

In [92]:
cadena.invoke({"question":"Cuál es el tiempo de entrega promedio para cada tipo de clima?"})

{'tool_output': clima
 Lluvioso    123.211825
 Nublado     138.286773
 Soleado     103.664453
 Tormenta    123.238509
 Name: tiempo_entrega, dtype: float64,
 'response': "```python\nimport pandas as pd; \ndf['tiempo_entrega'] = pd.to_numeric(df['tiempo_entrega']); \ntiempo_entrega_promedio = df.groupby('clima')['tiempo_entrega'].mean(); \nprint(tiempo_entrega_promedio)\n```"}

In [93]:
respuesta = cadena.invoke({"question":"Cuál es el tiempo de entrega promedio para cada tipo de clima?"})
print(respuesta['response'])

df.groupby('clima')['tiempo_entrega'].mean(); df.groupby('clima')['tiempo_entrega'].mean().to_frame('tiempo_entrega_promedio');


In [94]:
respuesta_2 = cadena.invoke({"question":"Cuál es el promedio de tiempo de entrega?"})
print(respuesta_2['response'])

df['tiempo_entrega'].mean();


In [95]:
respuesta_3 = cadena.invoke({"question":"""
                                        Cuál es la correlación entre los años de experiencia
                                        del colaborador y el tiempo de entrega? Es mayor que la
                                        correlación entre la clasificación del colaborador y el
                                        tiempo de entrega?
                                        """})
print(respuesta_3['response'])

Correlación entre años de experiencia y tiempo de entrega:  -0.2516410755060904
Correlación entre clasificación del colaborador y tiempo de entrega:  -0.29197537876110097
```python
correlacion_anios_experiencia = df['años_experiencia_colaborador'].corr(df['tiempo_entrega']); 
correlacion_clasificacion_colaborador = df['clasificacion_colaborador'].corr(df['tiempo_entrega']); 
print(correlacion_anios_experiencia > correlacion_clasificacion_colaborador)
```


In [98]:
correlacion_anios_experiencia = df['años_experiencia_colaborador'].corr(df['tiempo_entrega']);
correlacion_clasificacion_colaborador = df['clasificacion_colaborador'].corr(df['tiempo_entrega']);
print(correlacion_anios_experiencia > correlacion_clasificacion_colaborador)

True


In [99]:
respuesta_4 = cadena.invoke({"question":"""
                                        Cuál es la mediana de las columnas anteriormente mencionadas?
                                        """})
print(respuesta_4['response'])

```python
import pandas as pd; 
mediana = df.median(); 
print(mediana)
```


In [101]:
from langchain_experimental.agents import create_pandas_dataframe_agent

agente_ejecutor = create_pandas_dataframe_agent(
    llm = llm,
    df = df,
    agent_type = 'tool-calling',
    verbose = True,
    allow_dangerous_code = True
)

agente_ejecutor.invoke({"input":"""
                                Cuál es la correlación entre los años de experiencia
                                del colaborador y el tiempo de entrega? Es mayor que la
                                correlación entre la clasificación del colaborador y el
                                tiempo de entrega?
                                """})



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])"}`


-0.2516410755060904
Invoking: `python_repl_ast` with `{'query': "df['clasificacion_colaborador'].corr(df['tiempo_entrega'])"}`


-0.29197537876110097
Invoking: `python_repl_ast` with `{'query': "print(df['años_experiencia_colaborador'].corr(df['tiempo_entrega']) > df['clasificacion_colaborador'].corr(df['tiempo_entrega']))"}`


True
La correlación entre los años de experiencia del colaborador y el tiempo de entrega es -0.25, lo que indica una correlación negativa moderada. Esto significa que a medida que aumentan los años de experiencia del colaborador, el tiempo de entrega tiende a disminuir.

La correlación entre la clasificación del colaborador y el tiempo de entrega es -0.29, lo que indica una correlación negativa moderada. Esto significa que a medida que aumenta la clasificación del colaborador, el tiempo de entrega tiende a dismi

{'input': '\n                                Cuál es la correlación entre los años de experiencia\n                                del colaborador y el tiempo de entrega? Es mayor que la\n                                correlación entre la clasificación del colaborador y el\n                                tiempo de entrega?\n                                ',
 'output': 'La correlación entre los años de experiencia del colaborador y el tiempo de entrega es -0.25, lo que indica una correlación negativa moderada. Esto significa que a medida que aumentan los años de experiencia del colaborador, el tiempo de entrega tiende a disminuir.\n\nLa correlación entre la clasificación del colaborador y el tiempo de entrega es -0.29, lo que indica una correlación negativa moderada. Esto significa que a medida que aumenta la clasificación del colaborador, el tiempo de entrega tiende a disminuir.\n\nLa comparación entre las dos correlaciones muestra que la correlación entre los años de experiencia d

In [102]:
agente_ejecutor.invoke({"input":"""
                                Cuáles son las dimensiones del dataframe?
                                Cuáles son los nombres de las columnas y los tipos de datos?
                                """})




> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'print(df.shape)'}`


(43739, 16)

Invoking: `python_repl_ast` with `{'query': 'print(df.columns)'}`


Index(['ID_pedido', 'años_experiencia_colaborador',
       'clasificacion_colaborador', 'latitud_tienda', 'longitud_tienda',
       'latitud_entrega', 'longitud_entrega', 'fecha_pedido', 'hora_pedido',
       'hora_retirada', 'clima', 'trafico', 'vehiculo', 'area',
       'categoria_producto', 'tiempo_entrega'],
      dtype='object')

Invoking: `python_repl_ast` with `{'query': 'print(df.dtypes)'}`


ID_pedido                        object
años_experiencia_colaborador      int64
clasificacion_colaborador       float64
latitud_tienda                  float64
longitud_tienda                 float64
latitud_entrega                 float64
longitud_entrega                float64
fecha_pedido                     object
hora_pedido                      object
hora_retirada                    object
clima   

{'input': '\n                                Cuáles son las dimensiones del dataframe?\n                                Cuáles son los nombres de las columnas y los tipos de datos?\n                                ',
 'output': 'Las dimensiones del dataframe son (43739, 16), lo que significa que hay 43739 filas y 16 columnas.\n\nLos nombres de las columnas son:\n- ID_pedido\n- años_experiencia_colaborador\n- clasificacion_colaborador\n- latitud_tienda\n- longitud_tienda\n- latitud_entrega\n- longitud_entrega\n- fecha_pedido\n- hora_pedido\n- hora_retirada\n- clima\n- trafico\n- vehiculo\n- area\n- categoria_producto\n- tiempo_entrega\n\nLos tipos de datos de las columnas son:\n- ID_pedido: object\n- años_experiencia_colaborador: int64\n- clasificacion_colaborador: float64\n- latitud_tienda: float64\n- longitud_tienda: float64\n- latitud_entrega: float64\n- longitud_entrega: float64\n- fecha_pedido: object\n- hora_pedido: object\n- hora_retirada: object\n- clima: object\n- trafico: obje